# 02 — Modelagem e backtesting

Este notebook prevê vendas diárias em horizonte de 1 a 7 dias com validação de origem móvel. Compara persistência do último dia, sazonal ingênuo de 7 dias, regressão Ridge e Random Forest. Os modelos e métricas usam vendas observadas sem ruptura; `latent_demand` não participa do treinamento.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from demand_forecasting.backtesting import expanding_window_backtest
from demand_forecasting.metrics import summarize_metrics
from demand_forecasting.synthetic import generate_synthetic_demand

sns.set_theme(style="whitegrid")
panel = generate_synthetic_demand(random_state=42)
panel.head()

## Backtesting de origem móvel

Em cada origem, os rótulos de treinamento são limitados a datas iguais ou anteriores à origem. Para cada horizonte há um regressor direto; calendário futuro é conhecido, enquanto defasagens e médias móveis terminam no último dia observado na origem. Os primeiros 365 dias formam a janela inicial e as origens seguintes avançam 56 dias.

In [ ]:
predictions = expanding_window_backtest(
    panel, horizon=7, min_train_days=365, step_days=56,
    n_estimators=50, random_state=42,
)
print(f"Previsões avaliadas: {len(predictions):,}")
display(predictions.head())

## Métricas: agregado e por horizonte

MAE mede erro absoluto médio em unidades. WAPE normaliza o erro absoluto pela demanda observada total; viés positivo indica superprevisão. WAPE e viés normalizado não são definidos quando a demanda total do grupo é zero. Métricas agregadas podem esconder produtos difíceis, por isso também serão abertas por horizonte e produto.

In [ ]:
overall = summarize_metrics(predictions, groupby=("model",)).sort_values("WAPE")
by_horizon = summarize_metrics(predictions, groupby=("model", "horizon")).sort_values(["horizon", "WAPE"])
display(overall.round(3))
display(by_horizon.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.lineplot(data=by_horizon, x="horizon", y="WAPE", hue="model", marker="o", ax=axes[0])
axes[0].set(title="WAPE por horizonte", xlabel="Dias à frente", ylabel="WAPE")
by_product = summarize_metrics(predictions, groupby=("model", "product_id"))
sns.barplot(data=by_product, x="product_id", y="WAPE", hue="model", ax=axes[1])
axes[1].set(title="WAPE por produto", xlabel="Produto", ylabel="WAPE")
axes[1].tick_params(axis="x", rotation=45)
plt.show()
display(by_product.sort_values(["product_id", "WAPE"]).round(3))

## Inspecionar uma origem

Uma métrica agregada não substitui inspeção dos erros e das decisões de reposição. Custos de desperdício, margem, prazo de entrega e criticidade de ruptura devem ser adicionados antes de escolher um modelo operacional. O menor WAPE neste exemplo sintético não garante melhor desempenho em dados reais.

In [ ]:
chosen_product = predictions["product_id"].iloc[0]
example = predictions.loc[predictions["product_id"] == chosen_product]
latest_origin = example["origin_date"].max()
latest = example.loc[example["origin_date"] == latest_origin]
sns.lineplot(data=latest, x="horizon", y="y_true", marker="o", label="Realizado")
sns.lineplot(data=latest, x="horizon", y="y_pred", hue="model", marker="o")
plt.title(f"Origem {latest_origin.date()} — {chosen_product}")
plt.xlabel("Dias à frente")
plt.ylabel("Unidades")
plt.show()

## Cuidados e próximos passos

- O backtest exclui dias de ruptura conhecidos; no mundo real, a qualidade desse sinal é parte do problema.
- Defasagens são vendas observadas e podem refletir ruptura recente. Futuramente, deve-se incluir disponibilidade passada e decidir como recuperar demanda censurada.
- Ajuste de hiperparâmetros deve ocorrer dentro de uma validação temporal, nunca no período final de teste.
- Converta previsões em decisões somente após incorporar custos de falta e excesso de estoque, validade, lead time e restrições de pedido.